In [14]:
# pandas data 불러오기
import pandas as pd

redwine = pd.read_csv('winequality-red.csv',sep=';')
X, y = redwine.iloc[:, :-1], redwine.iloc[:, -1]


In [ ]:
# 훈련데이터세과 평가 데이터셋 나누기
from sklearn.model_selection import train_test_split
import torch

X_train, X_test, y_train, y_test = train_test_split(X, y)  # numpy 배열

print("before ", type(X_train) )
#  Dataframe-> tensor로 변경
#  numpy array 로  변경 후  tensor로
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
X_test  = torch.tensor(X_test.to_numpy(), dtype=torch.float32)

y_train = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_test  = torch.tensor(y_test.to_numpy(), dtype=torch.long)

print("after ", type(X_train) )

before  <class 'pandas.core.frame.DataFrame'>
after  <class 'torch.Tensor'>


In [18]:
#  모델 정의
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(11, 100),
    nn.Sigmoid(),
    
    nn.Linear(100, 200),  # Dense(200)
    nn.ReLU(),

    nn.Linear(200, 50),   # Dense(50)
    nn.Tanh(),

    nn.Linear(50, 10),    # Dense(10)
    nn.Softmax(dim=1)     # softmax
)

def forward(self, x):
        return self.model(x)



# 모델 구조 출력
print(model)

Sequential(
  (0): Linear(in_features=11, out_features=100, bias=True)
  (1): Sigmoid()
  (2): Linear(in_features=100, out_features=200, bias=True)
  (3): ReLU()
  (4): Linear(in_features=200, out_features=50, bias=True)
  (5): Tanh()
  (6): Linear(in_features=50, out_features=10, bias=True)
  (7): Softmax(dim=1)
)


In [ ]:
# 학습
import torch.optim as optim
import torch.nn as nn


#손실함수
criterion = nn.CrossEntropyLoss()

# optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

#예측값
outputs = model(X_train )
# loss 계산
loss = criterion(outputs, y_train)

# gradient 초기화
optimizer.zero_grad()

# backward
loss.backward()

# weight 업데이트
optimizer.step()

# accuracy 계산
accuracy = (predicted == y_train).float().mean()

print("accuracy:", accuracy.item())

accuracy: 0.3994995951652527


In [ ]:
def train(model, trian_loader, criterion, optimizer, epochs=300) :
    model.train() #모델 학습 모드
    for epoch in range(epochs) :
        running_loss = 0.0
        for data, target in train_loader: #  미니 배치단위로 데이터 로드
            optimizer.zero_grad()
            output = model(data)
            loss =criterion(output, target)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f'Epoch {epoch+1}, Loss : {running_loss/len(train_loader)}')



In [ ]:
def predict(model, test_loader):
    model.eval() # 모델 평가 모드 설정
    predictions = []
    with torch.no_grad():
        for data in test_loader :
            inputs = data[0]
            output = model(inputs)
            predicated_classes = output.argmax(dim=1, keepdim=True)
            predictions.append(predicted_classes)
    return torch.cat(predictions).numpy()

predictions = predict(model, test_loader)